# OpenAI Fine-Tuning
[![Open In Collab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langsmith-cookbook/blob/main/fine-tuning-examples/export-to-openai/fine-tuning-on-chat-runs.ipynb)

Once you've captured run traces from your deployment (production or beta), it's likely you'll want to use that data to
fine-tune a model. This walkthrough will show a quick way to do so.

Steps:

1. Query runs (optionally filtering by project, time, tags, etc.)
   - [Optional] Create a 'training' dataset to keep track of the data used for this model.
2. Convert runs to OpenAI messages or another format)
3. Fine-tune and use new model.

In [1]:
import os
import time
import truststore

# Use macOS system trust store (includes corporate CA certs)
truststore.inject_into_ssl()

os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_TRACING_V2"] = "true"

from langsmith import Client
client = Client()

# Groq client
import openai as openai_mod
from langsmith.wrappers import wrap_openai

groq_client = wrap_openai(openai_mod.Client(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    ))


## 1. Query runs

LangSmith saves traces for each runnable component in your LLM application. You can then query these runs in a variety of ways to construct your a training dataset. We will show a few common patterns below.

For examples of more 'advanced' filtering, check out the [filtering guide](https://docs.smith.langchain.com/tracing/faq/querying_traces) in the LangSmith docs.

**List all LLM runs for a specific project.**

The simplest query is just listing "llm" runs in your project (filtering out runs with errors). Below is an example where we list all LLM runs in the default project.

In [2]:
import datetime

project_name = "default"
run_type = "llm"
end_time = datetime.datetime.now()

runs = client.list_runs(
    project_name=project_name,
    run_type=run_type,
    error=False,
)
print(f"Found runs in project '{project_name}'")

Found runs in project 'default'


#### Filter by feedback

Depending on how you're fine-tuning, you'll likely want to filter out 'bad' examples (and want to filter in 'good' examples).

You can directly list by feedback! Usually you assign feedback to the root of the run trace, so we will use 2 queries.

In [3]:
from langsmith import traceable

@traceable(run_type="chain", name="joke_chain")
def joke_chain(input_text: str) -> str:
    time.sleep(3)
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": f"Tell a joke for:\n{input_text}"}
        ]
    )
    return response.choices[0].message.content

result = joke_chain("foo")
print(result)

# Wait for trace to flush, then add feedback to the root run
time.sleep(5)
root_runs = list(client.list_runs(
    project_name="default",
    filter='eq(name, "joke_chain")',
    execution_order=1,
))
if root_runs:
    client.create_feedback(root_runs[0].id, key="user_click", score=1)
    print(f"Added feedback to run {root_runs[0].id}")

Why did Foo go to the doctor? 
Because Foo had a "bar" problem (a play on the Foo Fighters band name and the phrase 'bar problem' but also referring to the type of bar in human problems).
Added feedback to run 019cd2a8-7015-7e10-99b8-b38b88bddc59


In [4]:
project_name = "default"
end_time = datetime.datetime.now()

runs_with_feedback = list(client.list_runs(
    project_name=project_name,
    execution_order=1,
    filter='and(eq(feedback_key, "user_click"), eq(feedback_score, 1))',
    error=False,
))
print(f"Found {len(runs_with_feedback)} runs with positive feedback")

Found 0 runs with positive feedback


Once you have these run ids, you can find the LLM run if it is a direct child of the root or if you
use a tag for a given trace.

In [5]:
llm_runs = []
for run in runs_with_feedback:
    child_runs = list(client.list_runs(
        project_name=project_name, run_type="llm", parent_run_id=run.id
    ))
    if child_runs:
        llm_runs.append(child_runs[0])

print(f"Found {len(llm_runs)} LLM child runs")
if llm_runs:
    print("Tags:", llm_runs[0].tags)

Found 0 LLM child runs


#### Filter by tags

It's common to have multiple chain types in a single project, meaning that the LLM calls may span multiple tasks and domains. Tags are a useful way to organize runs by task, component, test variant, etc, so you can curate a coherent dataset.

Below is a quick example. Please also reference the [Tracing FAQs](https://docs.smith.langchain.com/tracing/faq/customizing_trace_attributes#adding-metadata-and-tags-to-tracess) for more information on tagging.

In [6]:
import uuid

unique_tag = f"call:{uuid.uuid4()}"

@traceable(run_type="chain", name="joke_chain_tagged", tags=["my-tag"])
def joke_chain_tagged(input_text: str, extra_tags=None) -> str:
    time.sleep(3)
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": f"Tell a joke based on the following prompt:\n\nPrompt:{input_text}"}
        ]
    )
    return response.choices[0].message.content

# Call with unique tag for filtering
result1 = joke_chain_tagged("podcasting these days")
print(result1)

time.sleep(3)

# Second call with same tag
result2 = joke_chain_tagged("podcasting these days")
print(result2)

Why did the podcast go to therapy recently? 

It was feeling a bit "off-mic" and worried it was just a "skip" in its life's story. But after some sessions, it realized it was just a minor "buffer" in its communication with listeners and that it needed to "reboot" its approach. Now it's podcasting these days with a newfound appreciation for mental health.
Why did the podcast go to therapy? 

Because it was struggling to find its niche, and it was feeling a little muted.


In [8]:
time.sleep(5)

project_name = "default"
end_time = datetime.datetime.now()

runs = list(client.list_runs(
    project_name=project_name,
    execution_order=1,
    filter='has(tags, "my-tag")',
))
print(f"Found {len(runs)} runs with tag 'my-tag'")

Found 2 runs with tag 'my-tag'


#### Filter by run name.

By default, the run name is the class of the object being traced. You can filter by run name to narrow your search by, e.g., the LLM class.

Below, we will list all runs sent to a "ChatAnthropic" llm.

In [9]:
# Filter by LLM run name - with Groq these show up as "ChatCompletion" via wrap_openai
project_name = "default"
run_type = "llm"

llm_runs_by_name = list(client.list_runs(
    project_name=project_name,
    run_type=run_type,
    error=False,
))
print(f"Found {len(llm_runs_by_name)} LLM runs")

Found 41 LLM runs


#### Retrieve prompt inputs directly

If you fetch the LLM or chat run directly, the input will be the formatted prompt, with the values injected. You may want to separate the injected values from the prompt templating to remove or reduce the quantity of instruction prompting needed to obtain the desired prediction.

If your chain is composed as runnables (for instance, if you use LangChain Expression Language),
each __prompt__ runnable will be given its own run trace. You can fetch the inputs to the prompt template directly so that when you fine-tune, you can elide the other template content and train directly on the input values and LLM outputs.

Take the following chain, for instance, which is promoted to a [RunnableSequence](https://api.python.langchain.com/en/latest/runnables/langchain_core.runnables.base.RunnableSequence.html) via the piping operation.

In [10]:
# Example: retrieve prompt inputs directly
@traceable(run_type="chain", name="summarizer_chain")
def summarizer_chain(input_text: str) -> str:
    time.sleep(3)
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": f"Summarize the following chat log: {input_text}"}
        ]
    )
    return response.choices[0].message.content

result = summarizer_chain("hi there, hello....")
print(result)

There is no chat log provided.


In [11]:
time.sleep(5)
project_name = "default"
end_time = datetime.datetime.now()

# List all LLM runs
all_llm_runs = list(client.list_runs(
    project_name=project_name,
    run_type="llm",
    end_time=end_time,
    error=False,
))
print(f"Found {len(all_llm_runs)} LLM runs total")

Found 42 LLM runs total


In [12]:
# Get LLM runs and extract inputs/outputs
collected = []
for llm_run in all_llm_runs[:5]:  # Take first 5
    if llm_run.inputs and llm_run.outputs:
        collected.append({"inputs": llm_run.inputs, "outputs": llm_run.outputs})

print(f"Collected {len(collected)} runs with inputs/outputs")

Collected 5 runs with inputs/outputs


## 1a: (Recommended) Add to a training dataset

While not necessary for the fast-path of making your first fine-tuned model, datasets help build in a principled way by helping track the exact data used in a given model. They also are a natural place to add manual review or spot checking in the web app.

In [13]:
ft_ds_name = "Fine-Tuning Dataset Example"
try:
    dataset = client.read_dataset(dataset_name=ft_ds_name)
    print(f"Dataset '{ft_ds_name}' already exists.")
except Exception:
    dataset = client.create_dataset(
        dataset_name=ft_ds_name,
        description=f"LLM runs from project {project_name} for fine-tuning demo",
    )
    print(f"Created dataset '{ft_ds_name}'.")

# Add LLM runs as examples
added = 0
for llm_run in all_llm_runs[:10]:
    if not llm_run.inputs or not llm_run.outputs:
        continue
    try:
        client.create_example(
            inputs=llm_run.inputs,
            outputs=llm_run.outputs,
            dataset_id=dataset.id,
        )
        added += 1
    except Exception:
        pass  # Skip duplicates

print(f"Added {added} examples to dataset.")

Created dataset 'Fine-Tuning Dataset Example'.
Added 10 examples to dataset.


## 2. Convert examples to chat messages format

We'll convert the stored examples back to OpenAI-style message dicts suitable for fine-tuning.

In [15]:
def extract_role_content(msg):
    """Extract role and content from either LangChain serialized or plain dict messages."""
    if isinstance(msg, dict):
        # Plain dict (Groq-style): {"role": "user", "content": "..."}
        if "role" in msg and "content" in msg:
            return {"role": msg["role"], "content": msg["content"]}
        # LangChain serialized: {"lc": 1, "type": "constructor", "kwargs": {...}}
        if "kwargs" in msg:
            kwargs = msg["kwargs"]
            type_map = {"human": "user", "ai": "assistant", "system": "system"}
            role = type_map.get(kwargs.get("type", ""), "user")
            content = kwargs.get("content", "")
            return {"role": role, "content": content}
        # LangChain id-based: {"id": ["langchain", ..., "HumanMessage"], ...}
        if "id" in msg and isinstance(msg["id"], list):
            name = msg["id"][-1] if msg["id"] else ""
            role = "user" if "Human" in name else "assistant" if "AI" in name else "system"
            content = msg.get("kwargs", {}).get("content", "")
            return {"role": role, "content": content}
    return {"role": "user", "content": str(msg)}


def convert_example_to_messages(example) -> dict:
    """Convert a LangSmith example to OpenAI message format."""
    inp = example.inputs or {}
    out = example.outputs or {}
    result_msgs = []

    # Handle inputs - could be {"messages": [...]} or other structures
    input_msgs = inp.get("messages", [])
    if isinstance(input_msgs, list) and len(input_msgs) > 0:
        # Could be nested list [[msg1, msg2]] or flat [msg1, msg2]
        if isinstance(input_msgs[0], list):
            input_msgs = input_msgs[0]
        for m in input_msgs:
            result_msgs.append(extract_role_content(m))

    # Handle outputs - could be Groq format or LangChain generations
    choices = out.get("choices", [])
    generations = out.get("generations", [])
    if choices:
        choice = choices[0] if isinstance(choices, list) else choices
        assistant_msg = choice.get("message", {})
        if assistant_msg:
            result_msgs.append({"role": "assistant", "content": assistant_msg.get("content", "")})
    elif generations:
        gen = generations[0] if isinstance(generations, list) else generations
        if isinstance(gen, list) and gen:
            gen = gen[0]
        msg_data = gen.get("message", gen)
        if isinstance(msg_data, dict) and "kwargs" in msg_data:
            content = msg_data["kwargs"].get("content", "")
        else:
            content = gen.get("text", str(gen))
        result_msgs.append({"role": "assistant", "content": content})

    return {"messages": result_msgs}

# Quick check
ex = next(client.list_examples(dataset_name=ft_ds_name), None)
if ex:
    converted = convert_example_to_messages(ex)
    for m in converted["messages"]:
        print(f"  {m['role']}: {m['content'][:80]}...")

  user: You are given some context (a premise) and a question (a hypothesis). You must i...
  assistant: Context: The person who is not allergic likes fruit but prefers not to eat veget...


In [16]:
all_messages = [
    convert_example_to_messages(example)
    for example in client.list_examples(dataset_name=ft_ds_name)
]
print(f"Converted {len(all_messages)} examples to message format.")
for m in all_messages[:2]:
    print(m)

Converted 10 examples to message format.
{'messages': [{'role': 'user', 'content': "You are given some context (a premise) and a question (a hypothesis). You must indicate with Yes/No answer whether we can logically conclude the hypothesis from the premise.\n\n---\n\nFollow the following format.\n\nContext: ${context}\n\nQuestion: ${question}\n\nReasoning: Let's think step by step in order to ${produce the answer}. We ...\n\nAnswer: Yes or No\n\n---\n\nContext: It is not a surprise that the skier noticed the large tree dragging behind him, but not the rock in front of him.\n\nQuestion: Can we logically conclude for sure that it is not a surprise that the skier noticed the large magnolia dragging behind him, but not the rock in front of him?\n\nReasoning: Context: It is not a surprise that the skier noticed the large tree dragging behind him, but not the rock in front of him.\n\nQuestion: Can we logically conclude for sure that it is not a surprise that the skier noticed the large magno

Now we have message dicts in OpenAI format. We can export them as JSONL for fine-tuning on any compatible platform.

In [17]:
# Clean up messages - keep only role & content, filter out function_call entries
finetuning_messages = []
for group in all_messages:
    cleaned = []
    skip = False
    for msg in group.get("messages", []):
        if "function_call" in msg:
            skip = True
            break
        cleaned.append({"role": msg.get("role", "user"), "content": msg.get("content", "")})
    if not skip and cleaned:
        finetuning_messages.append({"messages": cleaned})

print(f"Prepared {len(finetuning_messages)} fine-tuning examples.")

Prepared 10 fine-tuning examples.


## 3. Export as JSONL

Groq's free tier does not support fine-tuning, but we can export the data as JSONL in OpenAI fine-tuning format.
This file can be used with OpenAI, Together AI, Fireworks, or any provider that supports the same format.

In [18]:
import json

output_path = "finetuning_data.jsonl"
with open(output_path, "w") as f:
    for group in finetuning_messages:
        f.write(json.dumps(group) + "\n")

print(f"Wrote {len(finetuning_messages)} examples to {output_path}")

# Preview first entry
with open(output_path) as f:
    first_line = f.readline()
    print("\nFirst example:")
    print(json.dumps(json.loads(first_line), indent=2))

Wrote 10 examples to finetuning_data.jsonl

First example:
{
  "messages": [
    {
      "role": "user",
      "content": "You are given some context (a premise) and a question (a hypothesis). You must indicate with Yes/No answer whether we can logically conclude the hypothesis from the premise.\n\n---\n\nFollow the following format.\n\nContext: ${context}\n\nQuestion: ${question}\n\nReasoning: Let's think step by step in order to ${produce the answer}. We ...\n\nAnswer: Yes or No\n\n---\n\nContext: It is not a surprise that the skier noticed the large tree dragging behind him, but not the rock in front of him.\n\nQuestion: Can we logically conclude for sure that it is not a surprise that the skier noticed the large magnolia dragging behind him, but not the rock in front of him?\n\nReasoning: Context: It is not a surprise that the skier noticed the large tree dragging behind him, but not the rock in front of him.\n\nQuestion: Can we logically conclude for sure that it is not a surprise

To actually fine-tune, you would upload this JSONL to a provider that supports it:
- **OpenAI**: `openai.files.create(file=open("finetuning_data.jsonl","rb"), purpose="fine-tune")`
- **Together AI**: `together fine-tuning create --training-file finetuning_data.jsonl`
- **Fireworks AI**: Upload via their dashboard or CLI

For this demo, we'll instead verify the data by running one of our examples through the base Groq model.

In [19]:
# Verify: replay the first training example's input through Groq
if finetuning_messages:
    sample = finetuning_messages[0]["messages"]
    # Use all messages except the last (assistant response) as input
    input_messages = [m for m in sample if m["role"] != "assistant"]
    if not input_messages:
        input_messages = sample[:1]

    print("Input messages:", input_messages)
    time.sleep(3)
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=input_messages,
    )
    print(f"\nGroq response:\n{response.choices[0].message.content}")

Input messages: [{'role': 'user', 'content': "You are given some context (a premise) and a question (a hypothesis). You must indicate with Yes/No answer whether we can logically conclude the hypothesis from the premise.\n\n---\n\nFollow the following format.\n\nContext: ${context}\n\nQuestion: ${question}\n\nReasoning: Let's think step by step in order to ${produce the answer}. We ...\n\nAnswer: Yes or No\n\n---\n\nContext: It is not a surprise that the skier noticed the large tree dragging behind him, but not the rock in front of him.\n\nQuestion: Can we logically conclude for sure that it is not a surprise that the skier noticed the large magnolia dragging behind him, but not the rock in front of him?\n\nReasoning: Context: It is not a surprise that the skier noticed the large tree dragging behind him, but not the rock in front of him.\n\nQuestion: Can we logically conclude for sure that it is not a surprise that the skier noticed the large magnolia dragging behind him, but not the r

We can also check file line count and validate the JSONL format.

In [20]:
import json

with open("finetuning_data.jsonl") as f:
    lines = f.readlines()
    print(f"Total examples: {len(lines)}")
    for i, line in enumerate(lines):
        data = json.loads(line)
        roles = [m["role"] for m in data["messages"]]
        print(f"  Example {i+1}: {len(data['messages'])} messages, roles: {roles}")

Total examples: 10
  Example 1: 2 messages, roles: ['user', 'assistant']
  Example 2: 2 messages, roles: ['user', 'assistant']
  Example 3: 2 messages, roles: ['user', 'assistant']
  Example 4: 2 messages, roles: ['user', 'assistant']
  Example 5: 3 messages, roles: ['system', 'user', 'assistant']
  Example 6: 2 messages, roles: ['user', 'assistant']
  Example 7: 2 messages, roles: ['user', 'assistant']
  Example 8: 2 messages, roles: ['user', 'assistant']
  Example 9: 2 messages, roles: ['user', 'assistant']
  Example 10: 2 messages, roles: ['user', 'assistant']


## Conclusion

You've completed the end-to-end workflow:

1. **Traced** LLM runs using Groq via LangSmith
2. **Queried** runs by project, feedback, tags, and run name
3. **Created** a LangSmith dataset from traced runs
4. **Exported** the data as JSONL in OpenAI fine-tuning format

The exported `finetuning_data.jsonl` can be uploaded to any provider that supports OpenAI-format fine-tuning (OpenAI, Together AI, Fireworks, etc.).